# HOMEWORK 16 - Course Project

Done by: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/ComputerVision-PBY_HW16_Course_Project

- **Dataset:** BSD432 / CBSD432 (train), BSD68 / CBSD68 (test)
- **Тип задачі:** Тема 2 — Intelligent Denoising based on Autoencoders
- **Вимога курсового проєкту:** обов'язкове порівняння з фільтрацією Гауса
- **Узгодження з лектором (Ján Koloda):** затверджено напрям Denoising; схвалено датасети родини BSD та генерацію шуму «на льоту» як аугментацію
- **Мета:** дослідити та порівняти ефективність класичного бейслайну (фільтр Гауса, медіанний фільтр), згорткового автоенкодера (CNN) та спектрального підходу (Fourier KAN)

---

### Контекст задачі (Motivation)

Потрібно побудувати та дослідити пайплайн для інтелектуального усунення шуму із цифрових зображень (як чорно-білих, так і кольорових).

Відповідно до вимог теми 2 курсового проєкту, ключовими завданнями є:
- порівняння якості денойзингу відносно обов'язкового бейслайну Гауса (*Gaussian filtering*);
- дослідження стійкості алгоритмів до різних типів шуму (гаусівський, періодичний тощо);
- збереження дрібних деталей, контурів та текстур без ефекту «розмиття» (blurring);
- оцінка ефективності, швидкості обробки та сильних/слабких сторін кожного методу.

---
# Поетапний план реалізації проєкту (Roadmap)

## Етап 1 — Data Preparation & Loading (Підготовка та завантаження даних)

Що потрібно зробити:
- підготувати та структурувати каталоги для збереження зображень (`resources/data/`);
- завантажити/зчитати тренувальний набір (BSD432 / CBSD432) та тестовий бенчмарк (BSD68 / CBSD68);
- забезпечити універсальну підтримку як чорно-білих (монохромних), так і кольорових (RGB) зображень.

---

## Етап 2 — EDA & Noise Injection (Аналіз структури та контамінація шумом)

Що потрібно зробити:
- перевірити розмірності, орієнтацію та діапазон значень пікселів у датасеті;
- реалізувати генерацію різних типів шуму «на льоту» як аугментацію (гаусівський шум, періодичні смуги тощо);
- візуалізувати зашумлені зразки та їхній частотний спектр (2D FFT) для аналізу природи шуму.

---

## Етап 3 — Baseline 1: Spatial Filtering (Класична просторова фільтрація)

Що потрібно зробити:
- реалізувати методи просторової фільтрації без навчання:
  - медіанний фільтр для оцінки придушення локальних аномалій;
  - фільтр Гауса (обов'язковий бейслайн за вимогою курсового проєкту);
  - білатеральний фільтр для перевірки збереження меж;
- розрахувати базові метрики відновлення (MSE, MAE, PSNR, SSIM тощо).

---

## Етап 4 — Baseline 2: Convolutional Autoencoder (Базова згорткова нейромережа)

Що потрібно зробити:
- побудувати згортковий автоенкодер (Baseline CNN) із гнучкою функцією активації (SiLU / GELU / LeakyReLU / ELU тощо);
- налаштувати навчальний пайплайн на патчах $96 	imes 96$ з оптимізатором;
- проаналізувати криві збіжності функції втрат (Loss) та зафіксувати базову якість.

---

## Етап 5 — Hyperparameter Tuning: Baseline CNN (Підбір гіперпараметрів згорткової мережі)

Що потрібно зробити:
- визначити простір ключових параметрів для оптимізації базової згорткової мережі (активації, швидкість навчання тощо);
- виконати цільовий пошук оптимальної конфігурації через Optuna на підвибірці патчів для досягнення максимальної якості CNN;
- зафіксувати найкращі параметри та зберегти оптимізовану модель бейслайну.

---

## Етап 6 — Novel Model: Fourier KAN (Спектральний автоенкодер)

Що потрібно зробити:
- побудувати архітектуру Fourier KAN із нелінійними вагами на базі гармонійних ядер;
- провести базове навчання моделі за ідентичних умов (той самий датасет і кількість епох для чесного порівняння);
- проаналізувати початкову динаміку навчання та спектральні особливості.

---

## Етап 7 — Hyperparameter Tuning: Fourier KAN (Підбір гіперпараметрів спектральної моделі)

Що потрібно зробити:
- визначити простір ключових параметрів для оптимізації спектральної моделі (гармонійні сітки, параметри оптимізації тощо);
- виконати цільовий пошук оптимальних конфігурацій через Optuna на підвибірці патчів для досягнення максимальної ефективності Fourier KAN;
- зафіксувати найкращі конфігурації та навчити фінальну спектральну модель.

---

## Етап 8 — Benchmarking & Evaluation (Зведена таблиця метрик)

Що потрібно зробити:
- протестувати всі три підходи на контрольному наборі BSD68 / CBSD68;
- розрахувати готові бібліотечні метрики: попіксельні помилки та якість сприйняття (MSE, RMSE, MAE, PSNR, SSIM тощо);
- виміряти швидкість обробки одного кадру (Latency в мс) та кількість параметрів кожної моделі;
- побудувати зведену форматовану таблицю порівняння через `Reporter`.

---

## Етап 9 — Visual & Spectral Analysis (Візуальна та спектральна діагностика)

Що потрібно зробити:
- побудувати радіальний графік спектра енергії (Radial Power Spectrum) для перевірки збереження високих частот і текстур;
- сформувати порівняльну візуальну сітку відновлених зображень;
- побудувати карти залишкових помилок (Residual Error Maps) для наочної демонстрації сильних і слабких сторін кожного підходу.

---

## Етап 10 — Conclusions & Discussion (Висновки та підсумки)

Що потрібно зробити:
- узагальнити отримані результати за основними критеріями оцінювання проєкту;
- сформувати чіткі підсумкові тези (ефективність методів, швидкість роботи, висновки щодо спектрального підходу).

In [2]:
# # Silent installation or update
#
# import sys
# from packaging.requirements import Requirement
# from importlib.metadata import PackageNotFoundError, version
#
# # Clean cache
# !"{sys.executable}" -m pip cache purge -q
#
# # Force updating
# package_update = [
#     "pip",
#     "scikit-learn",
#     "pandas",
# ]
#
# for package_name in package_update:
#     !"{sys.executable}" -m pip install -U "{package_name}" -q
#
# # Install missing packages
# package_array = [
#     "jinja2",
#     "ipywidgets",
#     "nbformat",
#     "numpy",
#     "matplotlib",
#     "opencv-contrib-python",
#     "torch",
#     "torchvision",
#     "scikit-image",
#     "optuna",
# ]
#
#
# def is_package_satisfied(req_str: str) -> bool:
#     try:
#         req = Requirement(req_str)
#         installed_ver = version(req.name)
#         if req.specifier:
#             return installed_ver in req.specifier
#         return True
#     except PackageNotFoundError:
#         return False
#
#
# for package_name in package_array:
#     if not is_package_satisfied(package_name):
#         !"{sys.executable}" -m pip install "{package_name}" -q


In [ ]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input data
# hm_version = 16
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/ComputerVision-PBY_HW{hm_version}_Course_Project.git"
# main_file_name_ua = f"Bohdan_Pinchuk_CV_HW{hm_version}_UA.ipynb"
# main_file_name = f"Bohdan_Pinchuk_CV_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone data
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name_ua, main_file_name, ".git", ".idea", ".DS_Store"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"
#
# # Note. To upload big files to GitHub, execute the next command in the terminal
# # 1. brew install git-lfs
# # 2. git lfs install
# # 3.1. git lfs track "*.safetensors"
# # 3.2. git lfs track "*.bin"

## ✅ Рішення: Етап 1 — Data Preparation & Loading (Підготовка та завантаження даних)

Що потрібно зробити:
- підготувати та структурувати каталоги для збереження зображень (`resources/data/`);
- завантажити/зчитати тренувальний набір (BSD432 / CBSD432) та тестовий бенчмарк (BSD68 / CBSD68);
- забезпечити універсальну підтримку як чорно-білих (монохромних), так і кольорових (RGB) зображень.